---
title: Model a flowing non-viscous fluid using QUICK-PDE
description: Use the QUICK-PDE function to model a flowing non-viscous fluid.
---


{/* cspell:ignore zlabel ylabel xlabel */}

# Model a flowing non-viscous fluid using QUICK-PDE

{/* cspell:ignore edgecolor, royalblue, rstride, cstride, Navier, Iterat, Fevals, HDES, PINN, ftarget */}

<Admonition type="note" title="Note">
Qiskit Functions are an experimental feature available only to IBM Quantum&reg; Premium Plan, Flex Plan, and On-Prem (via IBM Quantum Platform API) Plan users. They are in preview release status and subject to change.
</Admonition>

*Usage estimate: 10 minutes on a Heron r2 processor. (NOTE: This is an estimate only. Your runtime might vary.)*

Note that the execution time of this function is generally greater than 20 minutes, as we need to queue multiple times through the iterations, so you might want to split this tutorial in two sections: the first one, in which
you read through it and launch the jobs, and the second one a few hours later
(giving ample time for the jobs to complete), to work with the results of the jobs.

## Background

This tutorial seeks to teach on an introductory level how to use the QUICK-PDE
function to solve complex multi-physics problems on 156Q Heron R2 QPUs by using
ColibriTD's H-DES (Hybrid Differential Equation Solver).
The underlying algorithm is described in [the H-DES paper](https://arxiv.org/abs/2410.01130).
Note that this solver can also solve non-linear equations.

Multi-physics problems - including fluids dynamics, heat diffusion, and material
deformation, to name a few - can be ubiquitously described by Partial
Differential Equations (PDEs).

Such problems are highly relevant for various industries and constitute an
important branch of applied mathematics. However, solving non-linear
multivariate coupled PDEs with classical tools remains challenging due to the
requirement of an exponentially large amount of resources.

This function is appropriate for equations with increasing complexity and
variables, and is the first step to unlock possibilities that were once
considered intractable. To fully describe a problem modeled by PDEs, it is
necessary to know the initial and boundary conditions. These can strongly
change the solution of the PDE and the path to find their solution.

This tutorial teaches you how to:

1. Define the parameters of the initial condition function.
2. Adjust the qubit number (used to encode the function of the differential equation), depth, and shot number.
3. Run QUICK-PDE to solve the underlying differential equation.

## Requirements

Before starting this tutorial, be sure you have the following installed:

* Qiskit SDK v2.0 or later (`pip install qiskit`)
* Qiskit Functions Catalog (`pip install qiskit-ibm-catalog`)
* Matplotlib (`pip install matplotlib`)
* Access to the QUICK-PDE function. Fill out the [form to request access](https://forms.cloud.microsoft/e/3Wi9cbjQPK).

## Plans

The QUICK-PDE function is available in three plans: Small, Medium and Large. The following table gives the limits and the services of each plan.

| Size | Mode         | Iterations max (all stages)     | Shots max per job | Max qubits                    | Compute profile (vCPU x RAM Go) |
| ---- | ------------ | ------------------------------- | ----------------- | ----------------------------- | ------------------------------- |
| S    | Job mode     | 25                              | 10,000            | 15                            | 8x32                            |
| M    | Job mode     | 50                              | 50,000            | 35                            | 16x128              |
| L    | Session mode | Unlimited (up to session limit) | 100,000           | Unlimited (up to QPU limits)  | 48x768                          |

The maximum number of iterations applies to the sum of the iterations of all the stages of the optimizer chain.


## Setup

Authenticate using your [API key](http://quantum.cloud.ibm.com/) (for a proper login, use [the getting started guide](https://quantum.cloud.ibm.com/docs/en/guides/functions-get-started)) and select the function as follows:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
)

quick = catalog.load("colibritd/quick-pde")

## Step 1: Set properties of the problem to solve

This tutorial covers the user experience from two perspectives: the
physical problem determined by the initial conditions, and the algorithmic
component in solving a fluid dynamics example on a quantum computer.

Computational Fluid Dynamics (CFD) has a broad range of applications, and thus it is
important to study and solve the underlying PDEs. An important family of PDEs
are the Navier-Stokes equations, which are a system of nonlinear partial
differential equations describing the motion of fluids. They are highly relevant
for scientific problems and engineering applications.

Under certain conditions, the Navier-Stokes equations reduce to Burgers'
equation, a convection–diffusion equation describing phenomena occurring in
fluid dynamics, gas dynamics, and nonlinear acoustics, to name a few, by modeling
dissipative systems.

The one-dimensional version of the equation depends on two variables:
$t \in \mathbb{R}_{\geq 0}$ modeling the temporal dimension, $x \in \mathbb{R}$
representing the spatial dimension. The general form of the equation is called
the viscous Burgers' equation and reads:

$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2},$$

where $u(x,t)$ is the fluid speed field at a given position $x$ and time $t$, and $\nu$
is the viscosity of the fluid. Viscosity is an important property of a fluid
that measures its rate-dependent resistance to motion or deformation, and thus it
plays a crucial role in the determination of the dynamics of a fluid. When the
viscosity of the fluid is null ($\nu = 0$), the equation becomes a conservation
equation that can develop discontinuities (shock waves), due to the lack of its
internal resistance. In this case, the equation is called the inviscid Burgers'
equation and is a special case of nonlinear wave equation.

Strictly speaking, inviscid flows do not occur in nature, but when modeling
aerodynamic flow, due to the infinitesimal effect of transport, using an
inviscid description of the problem can be useful. Surprisingly, more than
70% of aerodynamic theory deals with inviscid flows.

This tutorial uses the inviscid Burgers' equation as a CFD example to solve
on IBM&reg; QPUs using QUICK-PDE, given by:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0.$$

The initial condition for this problem is set to a linear function:
$u(t=0,x) = ax + b$, $\text{with }a,b\in\mathbb{R}$,
where $a$ and $b$ are arbitrary constants influencing the shape of the
solution. You can adjust $a$ and $b$ and see how they influence the
solving process and the solution. For this choice of initial condition,
the equation admits the following analytical solution:

$$u(t, x) = \frac{ax + b}{at + 1},$$

which holds as long as no shock has formed, as in, for $t < -1/a$ when $a < 0$.

In [ ]:
job = quick.run(
    use_case="CFD_BURGER",
    physical_parameters={"a": 1.0, "b": 1.0},
)

In [ ]:
print(job.result())

## Step 2 (if needed): Optimize problem for quantum hardware execution

By default, the solver uses physically-informed parameters, which are initial circuit parameters for a given qubit number and depth from which our solver will start.

The shots are also part of the parameters with a default value, since fine-tuning them is important.

Depending on the configuration you're trying to solve, the algorithm's
parameters to achieve satisfactory solutions might need to be adapted; for example, it
can require more or fewer qubits per variable $t$ and $x$, depending on $a$ and
$b$. The following adjusts the number of qubits per function per
variable, the depth per function, and the number of shots.

You can also see how to specify the backend and the execution mode.

In addition, physically-informed parameters might steer the optimization process
in a wrong direction; in that case, you can disable it by setting the
`initialization` strategy to `"RANDOM"`.

### Choose the function size

The function runs with a compute size, `"S"`, `"M"` or `"L"`, set with the `function_size` argument of `run`. The size that your problem needs depends on the mode, the total number of iterations (the sum of the `max_iter` of all the stages of the optimizer chain), the maximum number of shots per job and the maximum number of qubits, within the limits of the plans. If the size is smaller than needed, the job fails with the error code 4600. If it is larger, the job runs and reports an event with the error code 4601.

The following function calculates the size from the same arguments as `run`, without submitting anything. When `optimizers` or `nb_qubits` are omitted, the default values of the use case are used, as in `run`.


In [ ]:
from typing import Any, Literal

Size = Literal["S", "M", "L"]


_SMALL = {"iterations": 25, "shots": 10_000, "qubits": 15}
_MEDIUM = {"iterations": 50, "shots": 50_000, "qubits": 35}
_MAX_SHOTS = 100_000

_MODES = ("job", "batch", "session")

_DEFAULT = {
    "CFD_EULER": {
        "nb_qubits": {"u": {"t": 2, "x": 2}, "g": {"t": 2, "x": 2}},
        "optimizers": [
            ("CMA", {"shot": 2500, "max_iter": 12}),
            ("CMA", {"shot": 4000, "max_iter": 18}),
            ("LS_TRF", {"shot": 10000, "max_iter": 30}),
        ],
    },
    "CFD_BURGER": {
        "nb_qubits": {"u": {"t": 2, "x": 2}},
        "optimizers": [
            ("CMA", {"shot": 3000, "max_iter": 40}),
            ("CMA", {"shot": 15000, "max_iter": 15}),
            ("CMA", {"shot": 50000, "max_iter": 10}),
        ],
    },
    "MD": {
        "nb_qubits": {"u": {"x": 3}, "sigma": {"x": 3}},
        "optimizers": [
            ("CMA", {"shot": 2500, "max_iter": 12}),
            ("CMA", {"shot": 4000, "max_iter": 18}),
            ("CMA", {"shot": 6000, "max_iter": 12}),
            ("LS_TRF", {"shot": 10000, "max_iter": 55}),
        ],
    },
    "BS_1A": {
        "nb_qubits": {"F_0": {"x_0": 3, "x_1": 3}},
        "optimizers": [
            ("CMA", {"shot": 5000, "max_iter": 12}),
            ("CMA", {"shot": 5000, "max_iter": 12}),
            ("LS_TRF", {"shot": 10000, "max_iter": 26}),
        ],
    },
    "BS_2A": {
        "nb_qubits": {"F_0": {"x_0": 3, "x_1": 2, "x_2": 2}},
        "optimizers": [
            ("CMA", {"shot": 5000, "max_iter": 12}),
            ("CMA", {"shot": 5000, "max_iter": 12}),
            ("LS_TRF", {"shot": 10000, "max_iter": 19}),
        ],
    },
    "BS_3A": {
        "nb_qubits": {"F_0": {"x_0": 3, "x_1": 2, "x_2": 2, "x_3": 2}},
        "optimizers": [
            ("CMA", {"shot": 5000, "max_iter": 18}),
            ("CMA", {"shot": 40000, "max_iter": 18}),
            ("LS_TRF", {"shot": 10000, "max_iter": 12}),
        ],
    },
}


def calculate_function_size(problem: Any, **kwargs: Any) -> Size:
    """Return the size ("S", "M" or "L") required to run the function."""
    metrics = _measure(problem, **kwargs)
    if _is_small(metrics):
        return "S"
    if _is_medium(metrics):
        return "M"
    return "L"


def _measure(problem: Any, **kwargs: Any) -> dict:
    """Reduce the problem to a dict of numbers that drives your resource usage."""
    args = {**problem, **kwargs} if isinstance(problem, dict) else {**kwargs, "use_case": problem}

    use_case = str(args.get("use_case") or "").upper()
    if use_case not in _DEFAULT:
        raise ValueError(
            f"cannot size use_case {args.get('use_case')!r}: expected one of {list(_DEFAULT)}"
        )

    mode = "job" if args.get("run_local") else str(args.get("mode") or "job").lower()
    if mode not in _MODES:
        raise ValueError(f"cannot size mode {args.get('mode')!r}: expected one of {list(_MODES)}")

    optimizers = args.get("optimizers")
    if optimizers is None:
        optimizers = _DEFAULT[use_case]["optimizers"]
    if not isinstance(optimizers, (list, tuple)) or not optimizers:
        raise ValueError("optimizers must be a non-empty list of (name, parameters) pairs")
    iterations, shots = 0, 0
    for i, stage in enumerate(optimizers):
        if not isinstance(stage, (list, tuple)) or len(stage) != 2 or not isinstance(stage[1], dict):
            raise ValueError(f"optimizers[{i}] must be a (name, parameters) pair")
        options = {str(k).lower(): v for k, v in stage[1].items()}
        shot, max_iter = options.get("shot"), options.get("max_iter")
        if not shot or not max_iter:
            raise ValueError(f"optimizers[{i}] must give 'shot' and 'max_iter'")
        iterations += int(max_iter)
        shots = max(shots, int(shot))
    if shots > _MAX_SHOTS:
        raise ValueError(f"shots per job ({shots}) exceeds the maximum of {_MAX_SHOTS}")

    nb_qubits = args.get("nb_qubits")
    if nb_qubits is None:
        nb_qubits = _DEFAULT[use_case]["nb_qubits"]
    qubits = max(sum(int(q) for q in per_var.values()) for per_var in nb_qubits.values())

    return {"mode": mode, "iterations": iterations, "shots": shots, "qubits": qubits}


def _is_small(metrics: dict) -> bool:
    """Determine if the metrics fit T-size 'S'."""
    return (
        metrics["mode"] != "session"
        and metrics["iterations"] <= _SMALL["iterations"]
        and metrics["shots"] <= _SMALL["shots"]
        and metrics["qubits"] <= _SMALL["qubits"]
    )


def _is_medium(metrics: dict) -> bool:
    """Determine if the metrics fit T-size 'M'."""
    return (
        metrics["mode"] != "session"
        and metrics["iterations"] <= _MEDIUM["iterations"]
        and metrics["shots"] <= _MEDIUM["shots"]
        and metrics["qubits"] <= _MEDIUM["qubits"]
    )


Calculate the size of the second job with its qubits and its optimizer chain, then pass it to `run`:


In [ ]:
nb_qubits = {"u": {"t": 2, "x": 1}}
optimizers = [
    ("CMA", {"shot": 2500,  "max_iter": 12, "sigma": 0.5}),
    ("CMA", {"shot": 4000,  "max_iter": 18, "sigma": 0.15}),
    ("LS_TRF", {"shot": 10000, "max_iter": 30, "psr": True}),
]

size = calculate_function_size("CFD_BURGER", nb_qubits=nb_qubits, optimizers=optimizers)
print(size)

job_2 = quick.run(
    use_case="CFD_BURGER",
    physical_parameters={"a": 0.5, "b": 0.25},
    nb_qubits=nb_qubits,
    depth={"u": 3},
    optimizers=optimizers,
    function_size=size,
)

In [ ]:
print(job_2.result())

## Step 3: Compare the algorithm performances

You can compare the convergence process of our solution (HDES) of job_2 to the performance of a physics-informed neural networks (PINN) algorithm and solver (see the [paper](https://arxiv.org/abs/1711.10561) and the associated [GitHub repository](https://github.com/314arhaam/burger-pinn)).

In the example of job_2's output (quantum-based approach), only  13 parameters (12 circuit parameters plus 1 scaling parameter) are optimized with the classical solver.
The convergence process is as follows:

```
optimizers:
   CMA: {'ftarget': np.float64(0.1), 'verb_disp': 10, 'maxiter': 100}
   CMA: {'ftarget': np.float64(0.005), 'verb_disp': 10, 'maxiter': 20}
   CMA: {'ftarget': np.float64(0.0025), 'verb_disp': 10, 'maxiter': 30}
   CMA: {'ftarget': np.float64(0.0005), 'verb_disp': 10, 'maxiter': 10}

500 shots
================== CMA =================
option:  {'ftarget': np.float64(0.1), 'verb_disp': 10, 'maxiter': 100}
0/100, loss: 0.02456641

1000 shots
================== CMA =================
option:  {'ftarget': np.float64(0.005), 'verb_disp': 10, 'maxiter': 20}
0/20, loss: 0.03641833
1/20, loss: 0.02461719
2/20, loss: 0.0283689
3/20, loss: 0.009898383
4/20, loss: 0.04454522
5/20, loss: 0.007019971
6/20, loss: 0.00811147
7/20, loss: 0.01592619
8/20, loss: 0.00764708
9/20, loss: 0.01401516
10/20, loss: 0.01767467
11/20, loss: 0.01220387

5000 shots
================== CMA =================
option:  {'ftarget': np.float64(0.0025), 'verb_disp': 10, 'maxiter': 30}
0/30, loss: 0.01024792
1/30, loss: 0.004343748
2/30, loss: 0.01450951
3/30, loss: 0.008591284
4/30, loss: 0.00266414
5/30, loss: 0.007923613
6/30, loss: 0.02023853
7/30, loss: 0.01031438
8/30, loss: 0.009513116
9/30, loss: 0.008132266
10/30, loss: 0.005787766
11/30, loss: 0.00390582

10000 shots
================== CMA =================
option:  {'ftarget': np.float64(0.0005), 'verb_disp': 10, 'maxiter': 10}
0/10, loss: 0.002386168
1/10, loss: 0.004024823
2/10, loss: 0.001311999
3/10, loss: 0.003433991
4/10, loss: 0.002339664
5/10, loss: 0.002978438
6/10, loss: 0.005458391
7/10, loss: 0.002026701
8/10, loss: 0.00207467
9/10, loss: 0.001947627
final_loss: 0.00151994463476429

```
That means a loss below 0.0015 can be reached after 28 iterations, and with optimizing only a few classical parameters.

Now we can compare the same to the PINN solution with the default configuration suggested by the paper using a gradient-based optimizer. The equivalent of our circuit with 13 parameters to be optimized is the neural network, which requires at least eight layers of 20 neurons, and thus involves optimizing 3021 parameters.
Then, the target loss is reached at Step 315, loss: 0.0014988397.

![Graph showing PINN data compared with the HDES-Qiskit function.](/docs/images/tutorials/colibritd-pde/pinn-data.avif)

Now, since we want to do a fair comparison, we should use the same optimizer in both cases.
The lowest number of iterations we found for 12 layers of 20 neurons = 4701 parameters:
```
(10_w,20)-aCMA-ES (mu_w=5.9,w_1=27%) in dimension 4701 (seed=351961)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     20 5.398521572351456e-02 1.0e+00 9.98e-03  1e-02  1e-02 0:02.3
    2     40 5.444650724530220e-02 1.0e+00 9.97e-03  1e-02  1e-02 0:05.1
    3     60 4.447407275438309e-02 1.0e+00 9.95e-03  1e-02  1e-02 0:08.2
    4     80 2.068969979882240e-02 1.0e+00 9.94e-03  1e-02  1e-02 0:11.7
    6    120 1.028892211616039e-02 1.0e+00 9.91e-03  1e-02  1e-02 0:20.1
    7    140 5.140972323715687e-03 1.0e+00 9.90e-03  1e-02  1e-02 0:25.4
    9    180 3.811701666563749e-03 1.0e+00 9.87e-03  1e-02  1e-02 0:37.4
   10    200 3.189878538250923e-03 1.0e+00 9.85e-03  1e-02  1e-02 0:44.2
   12    240 2.547040116041899e-03 1.0e+00 9.83e-03  1e-02  1e-02 0:59.7
   14    280 2.166548743844032e-03 1.0e+00 9.80e-03  1e-02  1e-02 1:18.0
   15    300 1.783065614290535e-03 1.0e+00 9.79e-03  1e-02  1e-02 1:28.4
   16    320 2.045844215899706e-03 1.0e+00 9.78e-03  1e-02  1e-02 1:39.8
Stopping early: loss 0.001405 <= target 0.0015
CMA-ES finished. Best loss: 0.001404788694344461
```

You can do the same with your data from job_2, and plot a comparison to the PINN solution.

In [ ]:
# check the loss function and compare between the two approaches
print(job_2.logs())

## Step 4: Use the result

With your solution, you can now choose what to do with it. The following demonstrates how to plot the result.

In [ ]:
solution = job.result()

# Plot the solution of the second simulation job_2
_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

Notice the difference of initial condition for the second run and its effect on
the result:

In [ ]:
solution_2 = job_2.result()

# Plot the solution of the second simulation job_2
_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution_2["samples"]["t"], solution_2["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution_2["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution_2["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

## Tutorial survey

Please take a minute to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience:

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_3BLFkNVEuh0QBWm)